# Loading and Inspecting the Data

The shared path helper locates local `data/raw/` inputs or Kaggle competition data. Run this notebook inside the repository checkout, with its shared package installed or accessible through the setup cell. JSONL has one battle object per line.


In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd

# This notebook may run from the repo root or any notebooks/<name>/ folder.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/pokemon_challenge").is_dir()), None)
if ROOT is not None and str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from pokemon_challenge.paths import data_dir, output_dir

DATA_DIR = data_dir()
OUTPUT_DIR = output_dir("aimilia", "106feat")  # 106 features
print(f"Using data from: {DATA_DIR}")

train_file_path = DATA_DIR / "train.jsonl"
test_file_path = DATA_DIR / "test.jsonl"
sample_submission_path = DATA_DIR / "sample_submission.csv"


def load_jsonl(path):
    """Read a .jsonl file into a list of dicts, one per line."""
    with open(path) as f:
        return [json.loads(line) for line in f]


train_data = load_jsonl(train_file_path)
test_data = load_jsonl(test_file_path)
print(f"Loaded {len(train_data)} train and {len(test_data)} test battles.")

Using data from: /Users/emilykokkinofta/Projects/FDS_Challenge_26/data
Loaded 10000 train and 5000 test battles.


### Duplicates

Don't need to be removed ( removing them makes the CV go 0.0012 lower- not much but unnessecary)

# Features

In [2]:
from features_static import build_static
from features_timeline import build_timeline
from features_effects import build_effects
from features_roster import build_roster

def build_features(data, labelled):
    ef = build_effects(data)
    boost_cols = ["battle_id"] + [c for c in ef.columns if "boost" in c]
    return (build_static(data, labelled=labelled)
            .merge(build_timeline(data), on="battle_id")
            .merge(ef[boost_cols], on="battle_id")
            .merge(build_roster(data), on="battle_id"))

train = build_features(train_data, labelled=True)
test  = build_features(test_data,  labelled=False)

features = [c for c in train.columns if c not in ("battle_id", "player_won")]
X_train, y_train = train[features], train["player_won"]
X_test = test[features]
print(f"{len(features)} features")     

106 features



### from build_static (features before the fight starts):

* p1:
    - six pokemons.
        - 2 summaries over stat: mean and max (6*2=12)
    - 2 levels (mean, min)
    - shape (types: how many elements the team covers, dual_type: hoe many creatures have two types instead of one, special_item: how many carry something other than the defaut item) -3

total = 17 features

* p2:
    - one summary over stat per pokemon: only one pokemon shown-no team to average (6)
    - lead pokemons extras ( level, types) -2

total = 8

* differences for the two teams (p1 summary minus p2 lead)
    - 2 for each stat :mean, max (6*2=12)

total = 12

grand total from build_static = 37 features


### from build_timeline ( things that happen in the 30 runs):
each of the 30 turns stores 4 things per side: active creatures, HP percentage of the active creature, its status condition, and which move it used.

We get each of the following twice ( one per team )
- faints: how many creatures died -2
- health: hppct_mean, hppct_final, hppct_min -6
- health over time: hppct_early_mean, hppct_late_mean, hppct_trend	health in turns 1–10, turns 21–30, and the change between -6
- status condition: status_rate, slp_rate -4
- loss of control: switches: how often they swapped creatures, n_species_seen: how many different ones appeared, no_move_rate: share of turns where no move happened (switching, fainted, asleep) -6
- move quality: mean_base_power, mean_accuracy: how hard and how reliably their moves hit, status_move_rate: share of non-damaging moves — a defensive playstyle -6
- differences: diff_hppct_mean   diff_hppct_final   diff_status_rate
diff_no_move_rate   diff_switches - 5

grand total for build_timeline:  35

### from build_effects (startegy duing the fight)- boost columns only:

we get each of the following twice (one per team):
- overall boost level: boost_mean (average total stages per turn),
  boost_max (highest total reached), boost_final (total at turn 30) -6
- boosts by kind: boost_offensive_mean (atk+spa), boost_defensive_mean
  (def+spd), boost_speed_mean (spe) -6
- difference: diff_boost_mean -1

grand total for build_effect= 13

### from features_roster ( notebook of every pokemon's health throughout the fights)

- 3 checkpoints, so the model can know how the health of the pokemons got there (turn 10-20-30)

    - check which pokemons are alive and the difference between the teams: (p1_alive, p2_alive, alive_diff) -3
    - the health of all pokemons added together and the difference (p1_teamhp, p2_teamhp, teamhp_diff) -3

total = 3x6= 18

- teamhp at turn 10 minus teamhp at turn 30: how fast each team is bleeding and the difference (p1_teamhp_decline, p2_teamhp_decline, teamhp_decline_diff) -3

grand total for features_roster: 21


GRAND TOTAL: 106 features (compared to 6 of the startup notebook)



# Model Comparison

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (GradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier)
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import LinearSVC


SEED = 42
cv_scheme = RepeatedStratifiedKFold(n_splits=5, n_repeats=6, random_state=0)

def scaled(estimator):
    """Scaling matters for distance- and penalty-based models; harmless for trees."""
    return make_pipeline(StandardScaler(), estimator)

candidates = {
    "Logistic Regression": scaled(LogisticRegression(max_iter=4000, random_state=SEED)),
    "Linear Discriminant": scaled(LinearDiscriminantAnalysis()),
    "Linear SVC":          scaled(LinearSVC(C=0.01, max_iter=5000, random_state=SEED)),
    "Gradient Boosting":   GradientBoostingClassifier(random_state=SEED),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=SEED),
    "Random Forest":       RandomForestClassifier(n_estimators=400, min_samples_leaf=5, random_state=SEED),
    "Extra Trees":         ExtraTreesClassifier(n_estimators=400, min_samples_leaf=5, random_state=SEED),
    "k-Nearest Neighbours": scaled(KNeighborsClassifier(n_neighbors=50)),
    "Decision Tree":       DecisionTreeClassifier(max_depth=6, min_samples_leaf=50, random_state=SEED),
}

results = {}
for name, candidate in candidates.items():
    scores = cross_val_score(candidate, X_train, y_train, cv=cv_scheme, n_jobs=-1)
    results[name] = scores

comparison = (pd.DataFrame({k: [v.mean(), v.std()] for k, v in results.items()}, index=["mean_accuracy", "std"]).T.sort_values("mean_accuracy", ascending=False))
display(comparison)


best_name = comparison.index[0]
print(f"Best model: {best_name} (CV {comparison.loc[best_name, 'mean_accuracy']:.4f})")

model = candidates[best_name]
model.fit(X_train, y_train)
test_predictions = model.predict(X_test)

,mean_accuracy,std
Gradient Boosting,0.664500,0.011108
Linear SVC,0.664083,0.010085
Logistic Regression,0.664017,0.010332
Linear Discriminant,0.663767,0.009512
Random Forest,0.662983,0.010622
Extra Trees,0.661983,0.011184
Hist Gradient Boosting,0.660033,0.010375
Decision Tree,0.642000,0.010104
k-Nearest Neighbours,0.633350,0.011273


Best model: Gradient Boosting (CV 0.6645)


# Creating the Submission File

The competition requires a `.csv` file with two columns: `battle_id` and `player_won`. Let's use our trained model to make predictions on the test set and format them correctly.

In [6]:
sample_submission = pd.read_csv(sample_submission_path)
expected_columns = ['battle_id', 'player_won']
if list(sample_submission.columns) != expected_columns:
    raise ValueError(f"Expected {expected_columns}, got {list(sample_submission.columns)}")

predictions_by_id = pd.Series(test_predictions, index=test["battle_id"])
submission_df = sample_submission.copy()
submission_df['player_won'] = submission_df['battle_id'].map(predictions_by_id)
if submission_df['player_won'].isna().any():
    raise ValueError("Some battle_id values have no matching prediction.")
submission_df['player_won'] = submission_df['player_won'].astype(int)

submission_df.to_csv(OUTPUT_DIR / 'submission.csv', index=False)

# record what produced this file: model, score, and feature count
best_scores = results[best_name]
(OUTPUT_DIR / "cv_score.txt").write_text(
    f"model: {best_name}\n"
    f"cv accuracy: {best_scores.mean():.4f} +/- {best_scores.std():.4f}\n"
    f"features: {len(features)}\n"
)
print(f"Saved: {OUTPUT_DIR / 'submission.csv'}")

Saved: /Users/emilykokkinofta/Projects/FDS_Challenge_26/outputs/aimilia/logreg-72feat/submission.csv


In [ ]:
#checking the submission file (if we have the corrrect one)

import datetime as dt

path = OUTPUT_DIR / "submission.csv"
check = pd.read_csv(path)
sample = pd.read_csv(sample_submission_path)

# 1. Is this file from the run you just did, or left over from an earlier one?
age = dt.datetime.now() - dt.datetime.fromtimestamp(path.stat().st_mtime)
print(f"written {age.total_seconds():.0f}s ago")

# 2. Shape and columns
assert list(check.columns) == ["battle_id", "player_won"], check.columns.tolist()
assert len(check) == len(sample) == 5000, len(check)

# 3. Exactly the right battles, in the right order
assert check["battle_id"].tolist() == sample["battle_id"].tolist(), "battle_id mismatch"
assert check["battle_id"].is_unique

# 4. Only 0s and 1s, no missing values
assert check["player_won"].isin([0, 1]).all(), check["player_won"].unique()
assert check["player_won"].notna().all()
assert check["player_won"].dtype.kind == "i", check["player_won"].dtype

# 5. Does the balance look sane? Training is 50/50, so a wild skew means trouble
share = check["player_won"].mean()
print(f"predicted 'player 1 wins' for {share:.1%} of battles")
assert 0.30 < share < 0.70, "suspicious class balance"

print("submission looks valid")

written 83s ago
predicted 'player 1 wins' for 50.6% of battles
submission looks valid
